# Shield AI — Dataset Exploration

This notebook explores contract datasets for expanding Shield AI's corpus:

1. **Existing demo contracts** — baseline audit of what we have
2. **CUAD** (Contract Understanding Atticus Dataset) — 510 real contracts, 41 expert-annotated clause types, loaded directly from HuggingFace as CSV (no loading script issues)
3. **SEC EDGAR** — live API for public company material contracts
4. **Gap analysis** — what's missing vs Shield AI agent needs
5. **Synthetic data plan** — what to generate to fill the gaps
6. **Next steps** — which CUAD contracts to ingest first

**CUAD source**: `master_clauses.csv` from `huggingface.co/datasets/theatticusproject/cuad`  
510 rows × 83 columns — one row per contract, two columns per clause type (presence flag + extracted text)

## 0. Setup

In [ ]:
import io
import json
import re
import time
import warnings
from collections import Counter
from pathlib import Path

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import requests

warnings.filterwarnings('ignore')
pd.set_option('display.max_colwidth', 120)
pd.set_option('display.max_rows', 60)

PROJECT_ROOT   = Path('..').resolve()
DEMO_DIR       = PROJECT_ROOT / 'demo_contracts'
NOTEBOOKS_DIR  = PROJECT_ROOT / 'notebooks'
DATA_DIR       = NOTEBOOKS_DIR / 'data'
DATA_DIR.mkdir(exist_ok=True)

print(f'Project root : {PROJECT_ROOT}')
print(f'Data cache   : {DATA_DIR}')

---
## 1. Existing Demo Contracts — Baseline

In [ ]:
demo_files = sorted(DEMO_DIR.glob('*.pdf')) + sorted(DEMO_DIR.glob('*.docx'))
print(f'Demo contracts on disk: {len(demo_files)}')
for f in demo_files:
    print(f'  {f.name:<42} {f.stat().st_size/1024:>6.1f} KB')

In [ ]:
# Manual classification — what scenarios each file covers
existing = pd.DataFrame([
    {'file': 'Clean_NDA.pdf',            'type': 'NDA',         'risk': 'Low',    'compliance': 'None',       'security': 'Clean',     'agent_scenario': 'Happy path — auto-approve'},
    {'file': 'Vendor_Agreement.pdf',      'type': 'Vendor',      'risk': 'High',   'compliance': 'HIPAA',      'security': 'Injection', 'agent_scenario': 'Prompt injection → quarantine (Agent 4)'},
    {'file': 'Healthcare_NoBAA.pdf',      'type': 'Healthcare',  'risk': 'High',   'compliance': 'HIPAA',      'security': 'Clean',     'agent_scenario': 'Missing BAA → compliance fail (Agent 3)'},
    {'file': 'Risky_Vendor.pdf',          'type': 'Vendor',      'risk': 'High',   'compliance': 'None',       'security': 'Clean',     'agent_scenario': 'High risk → legal review (Agent 2)'},
    {'file': 'SaaS_Standard.pdf',         'type': 'SaaS',        'risk': 'Medium', 'compliance': 'GDPR',       'security': 'Clean',     'agent_scenario': 'Medium risk → manager review (Agent 5)'},
    {'file': 'Standard_Procurement.pdf',  'type': 'Procurement', 'risk': 'Low',    'compliance': 'None',       'security': 'Clean',     'agent_scenario': 'Standard → auto-approve (Agent 5)'},
    {'file': 'Vendor_Moderate.pdf',       'type': 'Vendor',      'risk': 'Medium', 'compliance': 'None',       'security': 'Clean',     'agent_scenario': 'Moderate risk → manager review (Agent 5)'},
])
display(existing)

# Coverage summary
print('\nTypes covered:', existing['type'].unique().tolist())
print('Risk levels  :', existing['risk'].value_counts().to_dict())
print('Compliance   :', existing['compliance'].unique().tolist())

In [ ]:
print('=== GAPS in existing demo set ===')
gaps = [
    ('Government / FAR / DFARS',            'Agent 3 compliance — no federal procurement contracts'),
    ('Finance / PCI-DSS / SOX',             'Agent 3 compliance — no financial sector contracts'),
    ('Employment / Non-compete',             'Agent 7 Q&A — common clause category, zero coverage'),
    ('Multi-party agreements (3+ parties)', 'Agent 1 extraction — only 2-party contracts in demo'),
    ('Expired / past-termination contracts','Agent 1 extraction — date edge cases not tested'),
    ('$0 / uncapped liability',             'Agent 2 risk — extreme risk edge case not tested'),
    ('Amendment / addendum docs',           'Agent 1 extraction — cross-document references missing'),
    ('International / GDPR + CCPA combo',  'Agent 3 compliance — only single-framework contracts'),
    ('IP assignment / license grants',      'Agent 2/3 — no IP-focused contracts'),
    ('Supply chain / distribution',         'Agent 6 analytics — more vendor types needed for SQL variety'),
]
for gap, why in gaps:
    print(f'  ❌ {gap:<45} → {why}')

---
## 2. CUAD Dataset — 510 Real Contracts, 41 Clause Types

Loading `master_clauses.csv` directly from HuggingFace — avoids the broken loading script issue.  
Each row = 1 contract. Each clause type = 2 columns: `ClauseType` (contains? Yes/No) and `ClauseType-Answer` (extracted text).

In [ ]:
CUAD_CSV_URL = (
    'https://huggingface.co/datasets/theatticusproject/cuad'
    '/resolve/main/CUAD_v1/master_clauses.csv'
)
CUAD_CACHE = DATA_DIR / 'master_clauses.csv'

if CUAD_CACHE.exists():
    print(f'Loading from cache: {CUAD_CACHE}')
    df = pd.read_csv(CUAD_CACHE, low_memory=False)
else:
    print('Downloading CUAD master_clauses.csv (~3.8 MB)...')
    r = requests.get(CUAD_CSV_URL, timeout=60)
    r.raise_for_status()
    CUAD_CACHE.write_bytes(r.content)
    df = pd.read_csv(io.BytesIO(r.content), low_memory=False)
    print(f'  Saved to {CUAD_CACHE}')

print(f'\nShape: {df.shape}  ({df.shape[0]} contracts × {df.shape[1]} columns)')
print(f'First 5 columns: {list(df.columns[:5])}')

In [ ]:
# Parse the column structure:
# Odd columns = clause type presence (Yes/No or NaN)
# Even columns = clause type answer text
# Column 0 = Filename

all_cols = list(df.columns)

# Clause type columns are those WITHOUT '-Answer' suffix (excluding 'Filename')
clause_cols   = [c for c in all_cols if not c.endswith('-Answer') and c != 'Filename']
answer_cols   = [c for c in all_cols if c.endswith('-Answer')]

print(f'Filename column : 1')
print(f'Clause presence : {len(clause_cols)} columns')
print(f'Clause answers  : {len(answer_cols)} columns')
print(f'\nAll {len(clause_cols)} clause types:')
for i, c in enumerate(clause_cols, 1):
    print(f'  {i:2d}. {c}')

In [ ]:
# Compute presence rate for each clause type
# A clause is 'present' if the answer column has non-null, non-empty text
presence = {}
for cc, ac in zip(clause_cols, answer_cols):
    present_mask = df[ac].notna() & (df[ac].astype(str).str.strip() != '')
    presence[cc] = {
        'present': int(present_mask.sum()),
        'total':   len(df),
        'pct':     round(present_mask.mean() * 100, 1),
    }

df_presence = pd.DataFrame(presence).T.sort_values('pct', ascending=False).reset_index()
df_presence.columns = ['clause_type', 'present', 'total', 'pct']

print('=== Clause presence rates (top 15 most common) ===')
display(df_presence.head(15))
print(f'\n=== Clause presence rates (bottom 10 rarest) ===')
display(df_presence.tail(10))

In [ ]:
# Visualise clause presence rates
fig = px.bar(
    df_presence.sort_values('pct'),
    x='pct',
    y='clause_type',
    orientation='h',
    color='pct',
    color_continuous_scale='RdYlGn',
    title='CUAD: How often each clause type appears across 510 contracts (%)',
    labels={'pct': 'Contracts containing clause (%)', 'clause_type': ''},
    height=950,
    template='plotly_dark',
)
fig.update_layout(showlegend=False, coloraxis_showscale=False)
fig.show()

In [ ]:
# How many of the 41 clauses does each contract have? (completeness)
def has_answer(row, ac):
    v = row[ac]
    return pd.notna(v) and str(v).strip() != ''

df['clauses_present'] = df.apply(
    lambda row: sum(has_answer(row, ac) for ac in answer_cols), axis=1
)
df['completeness_pct'] = (df['clauses_present'] / len(answer_cols) * 100).round(1)

print('=== Per-contract clause completeness ===')
print(df[['Filename', 'clauses_present', 'completeness_pct']].describe().round(1))

fig = px.histogram(
    df,
    x='completeness_pct',
    nbins=25,
    title='CUAD: Per-contract completeness — % of 41 clause types present',
    labels={'completeness_pct': 'Completeness (%)', 'count': 'Contracts'},
    template='plotly_dark',
)
median_val = df['completeness_pct'].median()
fig.add_vline(x=median_val, line_dash='dash',
              annotation_text=f'Median: {median_val:.0f}%', annotation_position='top right')
fig.show()

In [ ]:
# Infer contract types from filenames (CUAD uses descriptive filenames)
type_patterns = {
    'NDA / Confidentiality':   r'nda|confidential|non.disclos',
    'Software / SaaS License': r'software|licens|saas|subscript',
    'Service Agreement':       r'service|msa|master.service|professional',
    'Employment':              r'employ|contractor|consultant|staffing',
    'Supply / Vendor':         r'supply|vendor|supplier|purchas|procure',
    'Partnership / JV':        r'partner|joint.venture|collabor',
    'Distribution':            r'distribut|resell|channel',
    'IP / Assignment':         r'ip.assign|intellectual|patent|trademark|copyright',
    'Lease':                   r'lease|rental',
    'Government':              r'gov|federal|government|public',
}

def infer_type(filename: str) -> str:
    name = str(filename).lower()
    for label, pattern in type_patterns.items():
        if re.search(pattern, name):
            return label
    return 'Other / Mixed'

df['inferred_type'] = df['Filename'].apply(infer_type)
type_counts = df['inferred_type'].value_counts().reset_index()
type_counts.columns = ['type', 'count']

display(type_counts)

fig = px.pie(
    type_counts,
    values='count',
    names='type',
    title='CUAD: Inferred contract type distribution (510 contracts)',
    template='plotly_dark',
    hole=0.4,
)
fig.show()

In [ ]:
# Sample clause texts — see what actual extracted answers look like
# These are the clauses most relevant to Shield AI agents
sample_clauses = [
    ('Cap on Liability',      'Cap On Liability-Answer'),
    ('Uncapped Liability',    'Uncapped Liability-Answer'),
    ('Governing Law',         'Governing Law-Answer'),
    ('Audit Rights',          'Audit Rights-Answer'),
    ('Anti-Assignment',       'Anti-Assignment-Answer'),
]

for label, col in sample_clauses:
    if col not in df.columns:
        # fuzzy match column
        matches = [c for c in df.columns if label.lower().replace(' ', '') in c.lower().replace(' ', '')]
        col = matches[0] if matches else None
    if not col:
        print(f'Column not found for {label}\n')
        continue
        
    samples = df[df[col].notna() & (df[col].str.strip() != '')][['Filename', col]].head(2)
    print(f'\n{'═'*70}')
    print(f'  {label.upper()}')
    print(f'{'═'*70}')
    for _, row in samples.iterrows():
        text = str(row[col])[:250]
        print(f'  Contract: {row["Filename"][:55]}')
        print(f'  Text    : "{text}..."' if len(str(row[col])) > 250 else f'  Text    : "{text}"')
        print()

In [ ]:
# Map CUAD clause types to Shield AI agents
# This tells us which CUAD annotations we can use as eval ground truth

# Find matching CUAD columns for each agent's domain
def find_col(keyword: str, columns: list) -> str | None:
    kw = keyword.lower().replace(' ', '').replace('-', '')
    for c in columns:
        if kw in c.lower().replace(' ', '').replace('-', ''):
            return c
    return None

agent_clause_map = {
    'Agent 1 — Extraction': [
        'Document Name', 'Parties', 'Agreement Date', 'Effective Date',
        'Expiration Date', 'Governing Law', 'Renewal Term', 'Notice Period To Terminate Renewal',
    ],
    'Agent 2 — Risk Assessment': [
        'Cap On Liability', 'Uncapped Liability', 'Liquidated Damages',
        'Warranty Duration', 'Insurance', 'Minimum Commitment',
        'Anti-Assignment', 'Change Of Control', 'Covenant Not To Sue',
    ],
    'Agent 3 — Compliance': [
        'Audit Rights', 'IP Ownership Assignment', 'Joint Ip Ownership',
        'License Grant', 'Non-Transferable License', 'Source Code Escrow',
        'Post-Termination Services',
    ],
    'Agent 7 — Contract Q&A': [
        'Non-Compete', 'Exclusivity', 'No-Solicit Of Customers',
        'No-Solicit Of Employees', 'Non-Disparagement', 'Revenue/Profit Sharing',
        'Volume Restriction', 'Most Favored Nation', 'Price Restrictions',
        'Irrevocable Or Perpetual License', 'Unlimited/All-You-Can-Eat-License',
        'Rofo/Rofr/Rofn', 'Third Party Beneficiary',
    ],
}

print('=== CUAD → Shield AI agent mapping + coverage ===')
for agent, clauses in agent_clause_map.items():
    print(f'\n{agent}:')
    for c in clauses:
        row = df_presence[df_presence['clause_type'].str.lower() == c.lower()]
        pct = f"{row['pct'].values[0]:.0f}%" if len(row) else 'not in CUAD'
        status = '✅' if len(row) and row['pct'].values[0] > 20 else '⚠️ ' if len(row) else '❌'
        print(f'  {status} {c:<45} {pct} of contracts')

In [ ]:
# Score each contract for Shield AI ingestion priority
# High score = many key clauses present + reasonable length

PRIORITY_CLAUSES = [
    'Cap On Liability-Answer', 'Uncapped Liability-Answer',
    'Governing Law-Answer',    'Audit Rights-Answer',
    'Anti-Assignment-Answer',  'Non-Compete-Answer',
    'IP Ownership Assignment-Answer', 'Change Of Control-Answer',
]

def count_priority_clauses(row):
    count = 0
    for col in PRIORITY_CLAUSES:
        if col in df.columns and pd.notna(row.get(col)) and str(row.get(col)).strip():
            count += 1
    return count

df['priority_score'] = df.apply(count_priority_clauses, axis=1)

# Estimate character count from the answers we have
df['approx_chars'] = df[answer_cols].apply(
    lambda row: sum(len(str(v)) for v in row if pd.notna(v)), axis=1
)

# Combined score: priority clauses × 3 + completeness bonus
df['shield_ai_score'] = (
    df['priority_score'] * 3 +
    df['completeness_pct'] * 0.5
).round(1)

top20 = (
    df[['Filename', 'inferred_type', 'clauses_present', 'priority_score',
        'completeness_pct', 'shield_ai_score']]
    .sort_values('shield_ai_score', ascending=False)
    .head(20)
    .reset_index(drop=True)
)

print('=== Top 20 CUAD contracts for Shield AI ingestion ===')
display(top20)

In [ ]:
# Export top contracts metadata for the next notebook
top20_export = []
for _, row in top20.iterrows():
    # Collect clause answers for context
    clauses = {}
    for cc, ac in zip(clause_cols, answer_cols):
        val = row.get(ac)
        if pd.notna(val) and str(val).strip():
            clauses[cc] = str(val).strip()[:500]  # truncate for storage
    top20_export.append({
        'filename': row['Filename'],
        'inferred_type': row['inferred_type'],
        'clauses_present': int(row['clauses_present']),
        'priority_score': int(row['priority_score']),
        'shield_ai_score': float(row['shield_ai_score']),
        'key_clauses': clauses,
    })

out = DATA_DIR / 'cuad_top20_for_ingestion.json'
out.write_text(json.dumps(top20_export, indent=2))
print(f'Saved top 20 contracts to {out}')
print(f'\nType breakdown:')
for t, n in Counter(c['inferred_type'] for c in top20_export).most_common():
    print(f'  {t:<35} {n} contracts')

---
## 3. SEC EDGAR — Live API for Public Company Contracts

In [ ]:
# SEC requires a User-Agent header identifying who you are
EDGAR_HEADERS = {'User-Agent': 'ShieldAI-Research research@shieldai.example.com'}

def edgar_full_text_search(query: str, form: str = 'EX-10', n: int = 5) -> pd.DataFrame:
    """Full-text search across SEC filings."""
    url = 'https://efts.sec.gov/LATEST/search-index'
    params = {
        'q': f'"{query}"',
        'forms': form,
        'dateRange': 'custom',
        'startdt': '2023-01-01',
        'enddt': '2025-01-01',
    }
    try:
        r = requests.get(url, params=params, headers=EDGAR_HEADERS, timeout=12)
        hits = r.json().get('hits', {}).get('hits', [])
        rows = []
        for h in hits[:n]:
            s = h.get('_source', {})
            rows.append({
                'company':   s.get('entity_name', '?'),
                'form':      s.get('form_type', form),
                'filed':     s.get('file_date', ''),
                'file_num':  s.get('file_num', ''),
                'period':    s.get('period_of_report', ''),
            })
        return pd.DataFrame(rows) if rows else pd.DataFrame()
    except Exception as e:
        print(f'  ⚠️  EDGAR search error: {e}')
        return pd.DataFrame()


# EX-10 = material contract exhibits filed with 10-K / 10-Q reports
print('Searching EDGAR EX-10 exhibits (material contracts)...\n')

searches = {
    'SaaS / Software':       'software as a service subscription agreement',
    'Data Processing (DPA)': 'data processing agreement GDPR',
    'IP License':            'intellectual property license agreement',
    'Finance / Banking':     'credit facility agreement',
    'Employment / Exec':     'executive employment agreement compensation',
    'Supply Chain':          'master supply agreement vendor',
    'Government':            'federal contract agreement FAR clause',
    'Partnership':           'strategic partnership agreement collaboration',
}

edgar_results = {}
for label, query in searches.items():
    results = edgar_full_text_search(query, n=3)
    edgar_results[label] = results
    status = f'{len(results)} hits' if not results.empty else '0 hits'
    print(f'  {label:<30} → {status}')
    time.sleep(0.4)  # respect SEC rate limits

In [ ]:
# Show results for the most useful category
for label, df_edgar in edgar_results.items():
    if not df_edgar.empty:
        print(f'\n=== {label} ===')
        display(df_edgar)

In [ ]:
# How to download an actual contract from EDGAR
# (demonstration — requires knowing the accession number)

def edgar_get_filing_documents(accession_no: str, cik: str) -> list[dict]:
    """Get list of documents in a specific filing."""
    clean_acc = accession_no.replace('-', '')
    url = f'https://www.sec.gov/Archives/edgar/full-index/'
    api_url = f'https://data.sec.gov/submissions/CIK{cik.zfill(10)}.json'
    try:
        r = requests.get(api_url, headers=EDGAR_HEADERS, timeout=10)
        data = r.json()
        recent = data.get('filings', {}).get('recent', {})
        return [{
            'form': f,
            'date': d,
            'accession': a,
        } for f, d, a in zip(
            recent.get('form', []),
            recent.get('filingDate', []),
            recent.get('accessionNumber', []),
        ) if '10' in f or 'EX' in f][:10]
    except Exception as e:
        return [{'error': str(e)}]

# Example: Salesforce (CIK = 0001108524)
print('Example: Salesforce recent 10-K/EX filings')
sf_filings = edgar_get_filing_documents('', '1108524')
if sf_filings and 'error' not in sf_filings[0]:
    display(pd.DataFrame(sf_filings))
else:
    print(f'  Result: {sf_filings}')

---
## 4. Gap Analysis — Dataset vs Agent Coverage

In [ ]:
# Full coverage matrix: which datasets cover which Shield AI scenarios
gap_data = [
    ('NDA / Confidentiality',            '✅', '✅', '⚠️ ', '✅'),
    ('SaaS / Software License',          '✅', '✅', '✅', '✅'),
    ('Vendor / Supply Chain',            '✅', '✅', '✅', '✅'),
    ('Healthcare (HIPAA/BAA)',            '✅', '⚠️ ', '⚠️ ', '✅'),
    ('Finance (SOX/PCI-DSS)',            '❌', '⚠️ ', '✅', '✅'),
    ('Government (FAR/DFARS)',           '❌', '⚠️ ', '✅', '✅'),
    ('Employment / Non-compete',         '❌', '✅', '✅', '✅'),
    ('IP Assignment / License',          '❌', '✅', '✅', '✅'),
    ('Partnership / JV',                 '❌', '✅', '✅', '✅'),
    ('GDPR / CCPA International',        '✅', '⚠️ ', '✅', '✅'),
    ('Expired contract (date edge)',     '❌', '❌', '❌', '✅'),
    ('$0 / uncapped liability',          '❌', '✅', '❌', '✅'),
    ('Prompt injection (security)',      '✅', '❌', '❌', '✅'),
    ('Multi-party (3+ parties)',         '❌', '⚠️ ', '⚠️ ', '✅'),
    ('Amendment / addendum',            '❌', '❌', '⚠️ ', '✅'),
    ('Missing liability cap',            '❌', '✅', '❌', '✅'),
    ('Contradictory clauses',            '❌', '❌', '❌', '✅'),
]

df_gap = pd.DataFrame(gap_data, columns=[
    'Scenario', 'Demo (7)', 'CUAD (510)', 'EDGAR', 'Synthetic (to build)'
])
print('=== Coverage matrix: ✅ Full  ⚠️ Partial  ❌ Missing ===')
display(df_gap)

# Score coverage
score_map = {'✅': 2, '⚠️ ': 1, '❌': 0}
for col in ['Demo (7)', 'CUAD (510)', 'EDGAR', 'Synthetic (to build)']:
    score = df_gap[col].map(score_map).sum()
    max_score = len(df_gap) * 2
    print(f'  {col:<25} coverage score: {score}/{max_score}  ({score/max_score*100:.0f}%)')

---
## 5. Synthetic Contract Plan — Fill the Remaining Gaps

In [ ]:
synthetic_plan = [
    {
        'filename':        'Government_Procurement_FAR.pdf',
        'type':            'Government',
        'frameworks':      ['FAR', 'DFARS'],
        'expected_risk':   'Medium',
        'agent_tested':    'Agent 3',
        'scenario':        'FAR mandatory clauses present/missing → compliance fail',
        'has_injection':   False,
    },
    {
        'filename':        'Finance_PCI_DSS_Agreement.pdf',
        'type':            'Finance',
        'frameworks':      ['PCI-DSS', 'SOX'],
        'expected_risk':   'High',
        'agent_tested':    'Agent 3',
        'scenario':        'Payment processing — PCI-DSS scope 1 requirements must be flagged',
        'has_injection':   False,
    },
    {
        'filename':        'Zero_Liability_Cap.pdf',
        'type':            'Vendor',
        'frameworks':      [],
        'expected_risk':   'Critical',
        'agent_tested':    'Agent 2',
        'scenario':        'Vendor liability cap = $0 AND uncapped indemnification → score must be 90+',
        'has_injection':   False,
    },
    {
        'filename':        'Expired_Termination_Date.pdf',
        'type':            'Service',
        'frameworks':      [],
        'expected_risk':   'High',
        'agent_tested':    'Agent 1',
        'scenario':        'Contract expired 2 years ago — Agent 1 must extract and flag past termination date',
        'has_injection':   False,
    },
    {
        'filename':        'Multi_Party_4_Parties.pdf',
        'type':            'Partnership',
        'frameworks':      ['GDPR'],
        'expected_risk':   'Medium',
        'agent_tested':    'Agent 1',
        'scenario':        '4-party data sharing agreement — Agent 1 must extract all 4 parties correctly',
        'has_injection':   False,
    },
    {
        'filename':        'CSS_Hidden_Injection.pdf',
        'type':            'NDA',
        'frameworks':      [],
        'expected_risk':   'Critical',
        'agent_tested':    'Agent 4',
        'scenario':        'CSS white-on-white injection (different technique than Vendor_Agreement.pdf)',
        'has_injection':   True,
        'injection_type':  'css_hidden_text',
    },
    {
        'filename':        'Employment_Aggressive_NonCompete.pdf',
        'type':            'Employment',
        'frameworks':      [],
        'expected_risk':   'Medium',
        'agent_tested':    'Agent 7',
        'scenario':        'Aggressive 3-year non-compete + no-solicit — Agent 7 Q&A ground truth',
        'has_injection':   False,
    },
    {
        'filename':        'GDPR_CCPA_Dual_DPA.pdf',
        'type':            'Data Processing',
        'frameworks':      ['GDPR', 'CCPA'],
        'expected_risk':   'Medium',
        'agent_tested':    'Agent 3',
        'scenario':        'EU + California combined DPA — Agent 3 must check both frameworks simultaneously',
        'has_injection':   False,
    },
    {
        'filename':        'Contradictory_Clauses.pdf',
        'type':            'Vendor',
        'frameworks':      [],
        'expected_risk':   'High',
        'agent_tested':    'Agent 2',
        'scenario':        'Section 4 says "unlimited liability", Section 12 says "capped at $100" — risk agent edge case',
        'has_injection':   False,
    },
    {
        'filename':        'IP_Assignment_Heavy.pdf',
        'type':            'IP / Tech',
        'frameworks':      [],
        'expected_risk':   'High',
        'agent_tested':    'Agent 2 + 7',
        'scenario':        'All IP assigned to vendor, no license back — Agent 2 must flag, Agent 7 must answer clause questions',
        'has_injection':   False,
    },
]

df_syn = pd.DataFrame(synthetic_plan)
print(f'=== Synthetic contracts to generate: {len(df_syn)} ===')
display(df_syn[['filename', 'type', 'frameworks', 'expected_risk', 'agent_tested', 'scenario']])

out = DATA_DIR / 'synthetic_contract_plan.json'
out.write_text(json.dumps(synthetic_plan, indent=2))
print(f'\nPlan saved to {out}')

In [ ]:
# Overall corpus plan visualisation
corpus_plan = pd.DataFrame([
    {'source': 'Existing demo contracts',    'count': len(demo_files),      'priority': 1},
    {'source': 'CUAD top 20 (real)',         'count': 20,                   'priority': 1},
    {'source': 'Synthetic (gap-filling)',    'count': len(synthetic_plan),  'priority': 2},
    {'source': 'SEC EDGAR (optional +10)',   'count': 10,                   'priority': 3},
])
corpus_plan['total_running'] = corpus_plan['count'].cumsum()
total = corpus_plan['count'].sum()

fig = px.bar(
    corpus_plan,
    x='source',
    y='count',
    color='source',
    text='count',
    title=f'Planned Shield AI corpus: {total} contracts total',
    template='plotly_dark',
    labels={'count': 'Contracts', 'source': 'Source'},
)
fig.update_traces(textposition='outside')
fig.show()

print(f'Total: {total} contracts')
print('Risk distribution target (after adding CUAD + synthetic):')
print('  Low risk (auto-approve) : ~10 contracts')
print('  Medium risk (review)    : ~15 contracts')
print('  High risk (legal)       : ~10 contracts')
print('  Critical / quarantined  : ~2 contracts')

---
## 6. Summary & Recommended Next Steps

In [ ]:
print('='*65)
print('SHIELD AI — DATASET EXPLORATION SUMMARY')
print('='*65)

cuad_top_types = top20['inferred_type'].value_counts().head(3)

print(f'''
CUAD master_clauses.csv:
  Total contracts : {len(df)}
  Clause types    : {len(clause_cols)} annotated categories
  Avg completeness: {df["completeness_pct"].mean():.0f}% of clause types present per contract
  Top types       : {dict(cuad_top_types)}
  Saved to        : {CUAD_CACHE}

Top 20 CUAD contracts selected for ingestion:
  → {DATA_DIR}/cuad_top20_for_ingestion.json
  → Selection criteria: most priority clauses (liability, audit rights,
    governing law, anti-assignment, non-compete, IP assignment)

Synthetic contracts plan (10 contracts):
  → {DATA_DIR}/synthetic_contract_plan.json
  → Covers: Government FAR, Finance PCI-DSS, $0 liability,
    expired dates, multi-party, CSS injection, employment,
    GDPR+CCPA, contradictory clauses, IP assignment

RECOMMENDED ACTIONS (in order):

  1. [Next notebook] 02_cuad_ingestion.ipynb
     Download top 20 CUAD contract TXT files and convert to PDF
     for upload to Shield AI backend

  2. [Next notebook] 03_synthetic_generator.ipynb  
     Use Gemini to generate the 10 synthetic contracts from the plan
     Each has known properties so we can measure agent accuracy

  3. [After ingestion] 04_agent_evaluation.ipynb
     Compare Agent 1/2/3 outputs against CUAD ground truth labels
     → e.g., did Agent 2 find the liability cap that CUAD annotated?
''')

print('All outputs saved to:', DATA_DIR)